# comfiui_ssh — ComfyUI + Cursor bridge

Working copy of `comfiui.ipynb` for live Cursor control.

Checkpoints and LoRAs are copied onto the VM disk before ComfyUI starts. Reading them through the Google Drive mount is what makes loads and folder scans take forever. Renders still save to Drive.

**Run order**

1. **Setup** — mount Drive, install ComfyUI, link `output` to Drive
2. **Stage models** — copy the checkpoints and LoRAs you list onto `/content/ComfyUI/models`
3. **Start ComfyUI**
4. **Public tunnel** — paste the `trycloudflare.com` URL into Cursor chat
5. **Log tail**

Colab's `*.colab.dev` link only works in *your* browser. Cursor needs the Cloudflare URL.

The checkpoint and LoRA menus only list files you staged. Add a path, re-run the stage cell, and restart ComfyUI to see new files.


In [ ]:
# @title
# =============================================================================
# ComfyUI on Google Colab + Drive + Public Pinggy Tunnel (shareable link)
# =============================================================================

# 0) Change to /content
%cd /content

# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2) Helper for shell commands
import sys, subprocess, os, time
py = sys.executable

def run(cmd, check=True):
    print(f"→ {cmd}")
    result = subprocess.run(cmd, shell=True, check=check, capture_output=True, text=True)
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print("Error (if any):", result.stderr.strip())
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
    return result

# 3) Upgrade pip
run(f"{py} -m pip install --upgrade pip")

# 4) Clone ComfyUI fresh
run("rm -rf ComfyUI", check=False)
run("git clone https://github.com/comfyanonymous/ComfyUI")

# 5) Install deps
%cd ComfyUI
# Install standard requirements (A100 supports standard PyTorch perfectly)
run(f"{py} -m pip install -r requirements.txt", check=False)

# Install torchsde
run(f"{py} -m pip install torchsde")

## Install the new ComfyUI-Manager pip package
run(f"{py} -m pip install -U comfyui-manager")

# Create ComfyUI-Manager config file to allow extension installation
MANAGER_CONFIG_DIR = "/content/ComfyUI/user/__manager"
os.makedirs(MANAGER_CONFIG_DIR, exist_ok=True)
with open(f"{MANAGER_CONFIG_DIR}/config.ini", "w") as f:
    f.write("[default]\n")
    f.write("network_mode = personal_cloud\n")
    f.write("security_level = weak\n") # Changed to weak for 0.0.0.0 access
print(f"✅ Created ComfyUI-Manager config: {MANAGER_CONFIG_DIR}/config.ini")

# 6) Renders stay on Drive. Models stay on this VM; the next cell copies them.
# A Drive symlink is a network filesystem, and ComfyUI's safetensors reads crawl on it.
DRIVE_BASE = "/content/drive/MyDrive/ComfyUI"
LOCAL_MODELS = "/content/ComfyUI/models"
os.makedirs(f"{DRIVE_BASE}/models", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/output", exist_ok=True)

output = "./output"
if os.path.lexists(output):
    run(f"rm -rf {output}")
run(f"ln -s {DRIVE_BASE}/output {output}")

if os.path.islink("models"):
    run("rm -rf models")
os.makedirs(LOCAL_MODELS, exist_ok=True)

print("\n✅ Paths:")
print(f"  Models (local) → {LOCAL_MODELS}")
print(f"  Output → {DRIVE_BASE}/output")
print("  Next cell copies checkpoints and LoRAs off Drive.")


In [ ]:
# @title Stage checkpoints and LoRAs onto the VM disk
# @markdown Relative to `MyDrive/ComfyUI/models`. One path per line, or several paths separated by ` | `.
# @markdown `checkpoints/` and `loras/` copy every file directly in those folders.
# @markdown Names in the skip box stay on Drive. A local copy of a skipped file is deleted.
# @markdown Re-running skips a file when the local copy is already the same size.
# @markdown The checkpoint and LoRA menus only list what this cell copied. Restart ComfyUI after adding files.

models_to_stage = "checkpoints/ | loras/ | text_encoders/gemma_3_12B_it_fp4_mixed.safetensors | latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors" # @param {type:"string"}
models_to_skip = "ltx2310eros_v12.safetensors | SexGod_Handjobs_LTX23_Rank64_v1.safetensors | LTX2_3_NSFW_furry_concat_v2.safetensors | DR34ML4Y_LTXXX_PREVIEW_RC1.safetensors | Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors | (ltx)cum_13_000002500.safetensors | epic_cumshots-V1-LTX_2_3-CUMSH0T.safetensors | Luxe_Sensual.safetensors | LTX2.3-MysticXXX.safetensors | Qwen-Edit-2509-Multiple-angles.safetensors" # @param {type:"string"}

import os
import shutil
import time

DRIVE_MODELS = "/content/drive/MyDrive/ComfyUI/models"
LOCAL_MODELS = "/content/ComfyUI/models"
COPY_BUFFER = 32 * 1024 * 1024


def parse_stage_paths(text):
    rels = []
    for raw_line in text.splitlines():
        for raw in raw_line.split("|"):
            line = raw.strip().replace("\\", "/").lstrip("/")
            if not line or line.startswith("#"):
                continue
            parts = [part for part in line.split("/") if part not in ("", ".")]
            if any(part == ".." for part in parts):
                raise ValueError(f"Path must stay inside models: {raw.strip()}")
            rels.append("/".join(parts))
    return rels


def list_stage_files(drive_models, rel):
    """Return files directly in rel, plus nested directory paths that were not copied."""
    root = os.path.abspath(drive_models)
    src = os.path.abspath(os.path.join(root, rel))
    if os.path.commonpath([src, root]) != root:
        raise ValueError(f"Path must stay inside models: {rel}")
    if not os.path.exists(src):
        raise FileNotFoundError(f"Not on Drive: {rel}")
    if os.path.isfile(src):
        return [(rel, src)], []
    found = []
    nested = []
    for name in sorted(os.listdir(src)):
        if name.startswith("."):
            continue
        path = os.path.join(src, name)
        child = f"{rel}/{name}" if rel else name
        if os.path.isfile(path):
            found.append((child, path))
        elif os.path.isdir(path):
            nested.append(child)
    return found, nested


def plan_copies(drive_models, local_models, rels):
    files = []
    notes = []
    for rel in rels:
        found, nested = list_stage_files(drive_models, rel)
        if not found:
            notes.append(f"no files directly in {rel}")
        for child in nested:
            notes.append(f"skipped nested folder {child} (add it as its own line to copy it)")
        files.extend(found)
    unique = []
    seen = set()
    for rel, src in files:
        if rel in seen:
            continue
        seen.add(rel)
        unique.append((rel, src))
    files = unique
    need = []
    skipped = []
    for rel, src in files:
        dst = os.path.join(local_models, rel)
        size = os.path.getsize(src)
        if os.path.isfile(dst) and os.path.getsize(dst) == size:
            skipped.append((rel, dst, size))
        else:
            need.append((rel, src, dst, size))
    return need, skipped, notes, files


def exclude_requested(files, skip_rels):
    """Drop files whose path or filename is listed in skip_rels."""
    skip = set(skip_rels)
    skip_names = {os.path.basename(item) for item in skip_rels}
    kept = []
    excluded = []
    matched = set()
    for rel, src in files:
        name = os.path.basename(rel)
        if rel in skip or name in skip or name in skip_names:
            excluded.append((rel, src))
            for item in skip_rels:
                if item in (rel, name) or os.path.basename(item) == name:
                    matched.add(item)
        else:
            kept.append((rel, src))
    unmatched = [item for item in skip_rels if item not in matched]
    return kept, excluded, unmatched


def remove_local_copies(local_models, excluded):
    removed = []
    for rel, _src in excluded:
        dst = os.path.join(local_models, rel)
        for path in (dst, dst + ".partial"):
            if os.path.isfile(path):
                os.remove(path)
                removed.append(os.path.relpath(path, local_models))
    return removed


def assert_free_space(path, bytes_needed):
    free = shutil.disk_usage(path).free
    if bytes_needed > free:
        raise RuntimeError(
            f"Not enough disk space on {path}. "
            f"Need {bytes_needed / 1e9:.2f} GB, free {free / 1e9:.2f} GB. "
            "Remove a path from the list or delete other files on the VM disk."
        )
    return free


def copy_buffered(src, dst, buf=COPY_BUFFER):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".partial"
    if os.path.exists(tmp):
        os.remove(tmp)
    total = os.path.getsize(src)
    copied = 0
    started = time.time()
    try:
        with open(src, "rb") as fin, open(tmp, "wb") as fout:
            while True:
                chunk = fin.read(buf)
                if not chunk:
                    break
                fout.write(chunk)
                copied += len(chunk)
                elapsed = max(time.time() - started, 1e-6)
                mb_s = (copied / (1024 * 1024)) / elapsed
                print(
                    f"\r  {copied / 1e9:.2f}/{total / 1e9:.2f} GB  {mb_s:.0f} MB/s",
                    end="",
                    flush=True,
                )
        os.replace(tmp, dst)
        print()
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


rels = parse_stage_paths(models_to_stage)
if not rels:
    raise RuntimeError("Add at least one checkpoint or LoRA path, one per line.")

need, skipped, notes, files = plan_copies(DRIVE_MODELS, LOCAL_MODELS, rels)
files, excluded, unmatched = exclude_requested(files, parse_stage_paths(models_to_skip))
excluded_rels = {rel for rel, _src in excluded}
need = [item for item in need if item[0] not in excluded_rels]
skipped = [item for item in skipped if item[0] not in excluded_rels]
for rel, src in excluded:
    notes.append(f"left on Drive {rel} ({os.path.getsize(src) / 1e9:.2f} GB)")
for item in unmatched:
    notes.append(f"skip list did not match a file: {item}")
for rel in remove_local_copies(LOCAL_MODELS, excluded):
    notes.append(f"deleted local {rel}")
for note in notes:
    print(note)
for rel, _dst, size in skipped:
    print(f"skip {rel} ({size / 1e9:.2f} GB already local)")

bytes_needed = sum(size for _rel, _src, _dst, size in need)
free = assert_free_space("/content", bytes_needed)
print(f"\nTo copy: {len(need)} file(s), {bytes_needed / 1e9:.2f} GB")
print(f"Free on /content: {free / 1e9:.2f} GB")

os.makedirs(LOCAL_MODELS, exist_ok=True)
for rel, src, dst, size in need:
    print(f"copy {rel} ({size / 1e9:.2f} GB)")
    copy_buffered(src, dst)

print("\n✅ Local models:")
print(f"  {LOCAL_MODELS}")
for rel, _src in files:
    dst = os.path.join(LOCAL_MODELS, rel)
    print(f"  {rel}  {os.path.getsize(dst) / 1e9:.2f} GB")


In [ ]:
# @title


import sys, subprocess, os
py = sys.executable

def run(cmd, check=True):
    print(f"→ {cmd}")
    result = subprocess.run(cmd, shell=True, check=check, capture_output=True, text=True)
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print("Error (if any):", result.stderr.strip())
    return result


# 2) Torch stack (CUDA 12.1). Keeping it simple; no xformers.
import sys, subprocess, os, time
py = sys.executable
def sh(cmd):
    print(cmd); subprocess.run(cmd, shell=True, check=True)

# 7) Launch ComfyUI
import sys, subprocess, os, time
py = sys.executable

# --- FIX FOR MANAGER SECURITY ERROR ---
# The Manager expects the settings under [default]
MANAGER_CONFIG_DIR = "/content/ComfyUI/user/__manager"
os.makedirs(MANAGER_CONFIG_DIR, exist_ok=True)
with open(f"{MANAGER_CONFIG_DIR}/config.ini", "w") as f:
    f.write("[default]\n")
    f.write("network_mode = personal_cloud\n")
    f.write("security_level = weak\n") # Changed to weak for 0.0.0.0 access
print("✅ Wrote fixed config.ini for ComfyUI-Manager (using [default] section)")

# # Kill any existing instances of ComfyUI to free the port
# os.system("pkill -f main.py")
# time.sleep(2)

PORT = 8188
LOG_FILE = "/content/comfyui.log"
LOCAL_MODELS = "/content/ComfyUI/models"
OUTPUT_DIR = "/content/drive/MyDrive/ComfyUI/output"

print(f"\n🚀 Starting ComfyUI on port {PORT} …")
print(f"   Logs: {LOG_FILE}")

log_file = open(LOG_FILE, "w", buffering=1)
comfy_proc = subprocess.Popen(
    [py, "-u", "main.py",
     "--enable-manager",
     "--listen", "0.0.0.0",
     "--port", str(PORT),
     "--enable-cors-header", "*",
     "--models-directory", LOCAL_MODELS,
     "--output-directory", OUTPUT_DIR],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd="/content/ComfyUI"
)

# 8) Get a direct Colab proxy URL (no tunnel services)
from google.colab import output as colab_output
import time
time.sleep(3)  # brief boot wait
proxy_url = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("\n🔗 OPEN THIS LINK:")
print(proxy_url)
print("\n📂 Models loaded from:", LOCAL_MODELS)
print("📂 Renders saved to:", OUTPUT_DIR)


In [ ]:
# Install cloudflared
run('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared')
run('chmod +x /usr/local/bin/cloudflared')
print('\n✅ cloudflared installed.')


In [ ]:
# @title
# 9) Start a public Cloudflare tunnel
# This gives a shareable URL that Cursor can access

print('\n⚡ Starting Cloudflare tunnel... (this may take a moment)')
# It might output some warnings, but it should still work.
# The tunnel will stop if this cell is stopped.

# Run the cloudflared daemon in the background
run('cloudflared tunnel --url http://127.0.0.1:8188 > /tmp/cloudflared.log 2>&1 &')

# Wait for the tunnel to establish and get the URL
time.sleep(5) # Give cloudflared a moment to start up

import re
tunnel_url = None
for _ in range(10): # Try a few times to find the URL in the log
    with open('/tmp/cloudflared.log', 'r') as f:
        log_content = f.read()
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', log_content)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(2)

if tunnel_url:
    print(f"\n🔗 Cloudflare Public Tunnel URL: {tunnel_url}")
    print("   **Paste this `trycloudflare.com` URL into Cursor chat.**")
else:
    print("\n⚠️ Could not find Cloudflare tunnel URL. Check /tmp/cloudflared.log for errors.")


In [ ]:
!tail -f /content/comfyui.log
